In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE FlashAttentionBwdFusedLse TEST (PURE TKV) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/flash_bwd_lse_cpu.cpu.c'; so=ex+'/flash_bwd_lse_cpu.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_flash' in txt, '(expect False)')
print('malloc/free =', 'malloc(' in txt and 'free(' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2500])

In [ ]:
# 3/4 - compare FlashAttentionBwdFusedLse (ctxo + dqo), ctx/lse from native fwd
import ctypes, numpy as np
P=ctypes.POINTER(ctypes.c_float); I=ctypes.c_int; D=ctypes.c_double
libn=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
fw=libn.FlashAttentionFwdLse; fw.argtypes=[P,I,P,I,P,I,I,I,I,I,D]; fw.restype=I
args=[P,I,P,I,P,I,P,I,P,I,P,I,I,I,I,I,D]
native=libn.FlashAttentionBwdFusedLse; native.argtypes=args; native.restype=I
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/flash_bwd_lse_cpu.so').FlashAttentionBwdFusedLse
tkv.argtypes=args; tkv.restype=I
rng=np.random.default_rng(59); inv=0.125
for n,h,head in [(64,256,0),(128,512,3),(256,1024,7),(65,192,1)]:
    qkv=rng.standard_normal(n*3*h).astype(np.float32)
    dctx=rng.standard_normal(n*h).astype(np.float32)
    ctx=np.zeros(n*64,np.float32); lse=np.zeros(n,np.float32)
    fw(qkv.ctypes.data_as(P),qkv.size,ctx.ctypes.data_as(P),ctx.size,lse.ctypes.data_as(P),lse.size,n,h,64,head,inv)
    oa=np.zeros(n*64,np.float32); oc=np.zeros(n*64,np.float32)
    qa=np.zeros(n*192,np.float32); qc=np.zeros(n*192,np.float32)
    native(qkv.ctypes.data_as(P),qkv.size,dctx.ctypes.data_as(P),dctx.size,ctx.ctypes.data_as(P),ctx.size,lse.ctypes.data_as(P),lse.size,oa.ctypes.data_as(P),oa.size,qa.ctypes.data_as(P),qa.size,n,h,64,head,inv)
    tkv  (qkv.ctypes.data_as(P),qkv.size,dctx.ctypes.data_as(P),dctx.size,ctx.ctypes.data_as(P),ctx.size,lse.ctypes.data_as(P),lse.size,oc.ctypes.data_as(P),oc.size,qc.ctypes.data_as(P),qc.size,n,h,64,head,inv)
    do=np.count_nonzero(oa.view(np.uint32)!=oc.view(np.uint32))
    dq=np.count_nonzero(qa.view(np.uint32)!=qc.view(np.uint32))
    print('n=%d h=%d head=%d ctxo_diff=%d/%d dqo_diff=%d/%d ctxo_maxabs=%.3g dqo_maxabs=%.3g'%(
        n,h,head,do,oa.size,dq,qa.size,np.abs(oa-oc).max(),np.abs(qa-qc).max()))

In [ ]:
# 4/4 - summary
print('Flash path tolerance waiver s0.125. Report maxabs for ctxo/dqo.')